In [ ]:

# ============================================================
# 📦 INSTALLS
# ============================================================

!pip install -q openai chromadb pypdf tiktoken ipywidgets

# ============================================================
# 📚 IMPORTS
# ============================================================

import uuid
import textwrap

from google.colab import files
from google.colab import userdata

from openai import OpenAI
from pypdf import PdfReader

import chromadb

# ============================================================
# 🔑 OPENAI CLIENT
# ============================================================

OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")

client = OpenAI(
    api_key=OPENAI_API_KEY
)

# ============================================================
# 📄 LOAD PDF
# ============================================================

def load_pdf(pdf_path):

    reader = PdfReader(pdf_path)

    text = ""

    for page in reader.pages:

        extracted = page.extract_text()

        if extracted:
            text += extracted + "\n"

    return text

# ============================================================
# ✂️ CHUNKER WITH OVERLAP
# ============================================================

def chunk_text(
    text,
    chunk_size=800,
    overlap=200
):

    chunks = []

    start = 0

    while start < len(text):

        end = start + chunk_size

        chunk = text[start:end]

        chunks.append(chunk)

        # move forward with overlap
        start += chunk_size - overlap

    return chunks

# ============================================================
# 🧠 CREATE EMBEDDING
# ============================================================

def get_embedding(text):

    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )

    return response.data[0].embedding

# ============================================================
# 🗂️ CHROMADB SETUP
# ============================================================

chroma_client = chromadb.Client()

collection = chroma_client.get_or_create_collection(
    name="simple_rag"
)

# ============================================================
# 📥 STORE CHUNKS
# ============================================================

def add_chunks_to_chroma(chunks):

    for chunk in chunks:

        embedding = get_embedding(chunk)

        collection.add(
            ids=[str(uuid.uuid4())],
            documents=[chunk],
            embeddings=[embedding]
        )

    print(f"Stored {len(chunks)} chunks in ChromaDB.")

# ============================================================
# 🔎 RETRIEVE DOCUMENTS
# ============================================================

def retrieve(query, top_k=3):

    query_embedding = get_embedding(query)

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )

    docs = results["documents"][0]

    return docs

# ============================================================
# 🤖 ASK QUESTION
# ============================================================

def ask(query):

    docs = retrieve(query)

    context = "\n\n".join(docs)

    prompt = f"""
You are a helpful RAG assistant.

Answer ONLY from the provided context.

If the answer is not in the context, say:
"I could not find the answer in the document."

Context:
{context}

Question:
{query}
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": "You answer questions using retrieved documents."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2
    )

    return response.choices[0].message.content

# ============================================================
# 📂 UPLOAD PDF
# ============================================================

print("Upload your PDF file.\n")

uploaded = files.upload()

pdf_path = list(uploaded.keys())[0]

print(f"\nLoaded PDF: {pdf_path}")

# ============================================================
# 🏗️ BUILD RAG PIPELINE
# ============================================================

print("\nLoading PDF...")

text = load_pdf(pdf_path)

print("PDF loaded.")

print("\nChunking document...")

chunks = chunk_text(
    text,
    chunk_size=800,
    overlap=200
)

print(f"Created {len(chunks)} chunks.")

print("\nGenerating embeddings and storing in ChromaDB...")

add_chunks_to_chroma(chunks)

print("\n✅ RAG Pipeline Ready")

# ============================================================
# 💬 INTERACTIVE CHATBOT
# ============================================================

print("\n🤖 Chatbot Ready")
print("Type 'exit' to stop.\n")

while True:

    query = input("You: ")

    if query.lower() == "exit":

        print("\nBot: Goodbye.")
        break

    answer = ask(query)

    print("\nBot:")

    print(textwrap.fill(answer, width=100))

    print("\n" + "=" * 100 + "\n")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 649.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.8/338.8 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 79.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/

Saving The_Brain_behind_AI (1).pdf to The_Brain_behind_AI (1).pdf

Loaded PDF: The_Brain_behind_AI (1).pdf

Loading PDF...
PDF loaded.

Chunking document...
Created 86 chunks.

Generating embeddings and storing in ChromaDB...
Stored 86 chunks in ChromaDB.

✅ RAG Pipeline Ready

🤖 Chatbot Ready
Type 'exit' to stop.

You: what is this document about

Bot:
This document is about the book titled *The Brain Behind AI: How Transformers Revolutionized
Artificial Intelligence*. It introduces the people involved in AI development and emphasizes the
importance of understanding AI as a human tool shaped by human values. The book explores the
evolution of AI, particularly focusing on the transformative impact of the transformer architecture,
and invites readers to engage with the technology that reflects human intelligence.


You: What is Transformer

Bot:
I could not find the answer in the document.


You: what is AI

Bot:
I could not find the answer in the document.


You: tell me About Transfor

In [ ]:
%pip install numpy pandas matplotlib seaborn scikit-learn